# Exploratory Data Analysis REES46 Heterogeneous Graph Dataset

Phiên bản mới data hiện tại. Notebook này nạp dữ liệu đã xử lý trực tiếp từ HuggingFace nguyenmaiductrong/rees46-bpatmp-temporal, đúng nguồn mà các notebook model MBGCN, MixRec, CRGCN, BPATMP đang dùng nên các số liệu mô tả dữ liệu khớp với phần thí nghiệm.

Khác với bản cũ: không có phần Spark quét lại raw CSV. Mục Temporal được thay bằng Temporal Split Composition dựng từ chính dữ liệu đã chia train, val, test.

Nội dung: 1 Thống kê graph cơ bản, 2 Phân phối tương tác class imbalance, 3 Phân phối bậc nút power law, 4 Cấu trúc split theo thời gian, 5 Bảng summary cho báo cáo.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from huggingface_hub import hf_hub_download

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Palatino', 'serif'],
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 0.8,
    'lines.linewidth': 1.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Nguồn dữ liệu: HuggingFace dataset giống các notebook model
REPO_ID = "nguyenmaiductrong/rees46-bpatmp-temporal"
REPO_TYPE = "dataset"

# Nếu dataset là private: đặt biến môi trường HF_TOKEN, hoặc bỏ comment 2 dòng dưới.
# from huggingface_hub import login
# login(os.environ.get("HF_TOKEN", ""))

def hf_path(filename):
    """Tải 1 file từ dataset trên HuggingFace, trả về đường dẫn local (đã cache)."""
    return hf_hub_download(
        REPO_ID,
        filename,
        repo_type=REPO_TYPE,
        token=os.environ.get("HF_TOKEN"),
    )

FIG_DIR = os.path.join(os.getcwd(), 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print('Repo:', REPO_ID)
print('Fig dir:', FIG_DIR)

## 1. Graph Basic Statistics

In [ ]:
# Node counts
with open(hf_path('node_counts.json')) as f:
    node_counts = json.load(f)

N_U = node_counts['user']
N_P = node_counts['product']
N_C = node_counts['category']
N_B = node_counts['brand']
N_total = N_U + N_P + N_C + N_B

# Behavioral edges train, nạp một lần và dùng lại cho phần degree
edges = {}
for beh in ['view', 'cart', 'purchase']:
    src = np.load(hf_path(f'{beh}_train_src.npy'), mmap_mode='r')
    dst = np.load(hf_path(f'{beh}_train_dst.npy'), mmap_mode='r')
    edges[beh] = (src, dst)

E_view_train = len(edges['view'][0])
E_cart_train = len(edges['cart'][0])
E_purchase_train = len(edges['purchase'][0])

# Eval pairs: val và test là các cặp purchase được giữ làm ground-truth
val_user_idx = np.load(hf_path('val_user_idx.npy'))
val_prod_idx = np.load(hf_path('val_product_idx.npy'))
test_user_idx = np.load(hf_path('test_user_idx.npy'))
test_prod_idx = np.load(hf_path('test_product_idx.npy'))
n_val = len(val_user_idx)
n_test = len(test_user_idx)

E_view = E_view_train
E_cart = E_cart_train
E_purchase = E_purchase_train + n_val + n_test

# Quan hệ heterogeneous Product to Category và Product to Brand, optional
E_belongsTo = E_producedBy = None
for fname, key in [
    ('node_mappings/product_category.parquet', 'belongsTo'),
    ('node_mappings/product_brand.parquet', 'producedBy'),
]:
    try:
        n = len(pd.read_parquet(hf_path(fname)))
        if key == 'belongsTo':
            E_belongsTo = n
        else:
            E_producedBy = n
    except Exception as e:
        print(f'skip {fname} {type(e).__name__} không có trong dataset, bỏ qua')

E_behavior_total = E_view + E_cart + E_purchase
max_pairs = N_U * N_P
sparsity = lambda e: 1.0 - e / max_pairs
eval_users_val = len(np.unique(val_user_idx))
eval_users_test = len(np.unique(test_user_idx))

print('Node Statistics')
print(f'User {N_U:,} Product {N_P:,} Category {N_C:,} Brand {N_B:,} Total {N_total:,}')
print('Edge Statistics (all splits combined)')
print(f'view {E_view:,} sparsity {sparsity(E_view):.6f}')
print(f'cart {E_cart:,} sparsity {sparsity(E_cart):.6f}')
print(f'purchase {E_purchase:,} sparsity {sparsity(E_purchase):.6f}')
if E_belongsTo is not None: print(f'belongsTo {E_belongsTo:,} Product to Category')
if E_producedBy is not None: print(f'producedBy {E_producedBy:,} Product to Brand')
print(f'behavior total {E_behavior_total:,}')
print('Temporal Split')
print(f'Val pairs {n_val:,} unique users {eval_users_val:,}')
print(f'Test pairs {n_test:,} unique users {eval_users_test:,}')

## 2. Interaction Distribution Class Imbalance

View Cart rất nhiều, Purchase rất thưa. Đây là lý do phải mô hình hóa multi behavior thay vì chỉ dùng tín hiệu purchase.

In [ ]:
behavior_counts = {'View': E_view, 'Cart': E_cart, 'Purchase': E_purchase}
total_beh = sum(behavior_counts.values())
behavior_pct = {k: v / total_beh * 100 for k, v in behavior_counts.items()}
for beh, cnt in behavior_counts.items():
    print(f'{beh} {cnt:,} {behavior_pct[beh]:.2f} percent')
print(f'Total {total_beh:,} 100.00 percent')

In [ ]:
labels = list(behavior_counts.keys())
counts = [behavior_counts[k] for k in labels]
pcts = [behavior_pct[k] for k in labels]
colors = ['#2c6fad', '#5ba4cf', '#a8d5f5']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))

bars = ax1.bar(labels, counts, color=colors, edgecolor='#1a1a1a', linewidth=0.8, zorder=3)
ax1.set_yscale('log')
ax1.set_ylabel('Number of Interactions log scale')
ax1.set_title('Behavior Interaction Count Log Scale')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, cnt, pct in zip(bars, counts, pcts):
    ax1.text(bar.get_x() + bar.get_width() / 2, cnt * 1.3,
             f'{cnt/1e6:.1f}M\n{pct:.1f} percent', ha='center', va='bottom', fontsize=9)
ax1.set_ylim(top=max(counts) * 15)
ax1.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)
sns.despine(ax=ax1)

wedges, _, autotexts = ax2.pie(
    counts, labels=None, autopct=lambda p: f'{p:.1f} percent', startangle=90, colors=colors,
    wedgeprops=dict(linewidth=0.8, edgecolor='white'), pctdistance=0.75, explode=[0, 0.04, 0.08])
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax2.legend(wedges, [f'{lbl} {cnt/1e6:.1f}M' for lbl, cnt in zip(labels, counts)],
           loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=1, frameon=False, fontsize=10)
ax2.set_title('Proportion of Behavior Types')

plt.suptitle('REES46 Dataset: Interaction Distribution across Behavior Types',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_interaction_distribution.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_interaction_distribution.png'))
plt.show()

## 3. Node Degree Distribution Power Law Property

Phân phối bậc long tail trên trục log log CCDF biện minh cho việc dùng phương pháp đồ thị và nêu vấn đề sparsity, cold start.

In [ ]:
# Gộp các hành vi, đếm bậc mỗi nút (trên train split)
user_src_all = np.concatenate([np.asarray(edges[b][0]) for b in ['view', 'cart', 'purchase']])
prod_dst_all = np.concatenate([np.asarray(edges[b][1]) for b in ['view', 'cart', 'purchase']])

user_degree = np.bincount(user_src_all, minlength=N_U)
prod_degree = np.bincount(prod_dst_all, minlength=N_P)
user_deg_nonzero = user_degree[user_degree > 0]
prod_deg_nonzero = prod_degree[prod_degree > 0]

print(f'User: active={len(user_deg_nonzero):,} mean={user_deg_nonzero.mean():.1f} median={np.median(user_deg_nonzero):.1f} max={user_deg_nonzero.max():,}')
print(f'Product: active={len(prod_deg_nonzero):,} mean={prod_deg_nonzero.mean():.1f} median={np.median(prod_deg_nonzero):.1f} max={prod_deg_nonzero.max():,}')

In [ ]:
# CCDF: xác suất degree ít nhất k, đường biên đẹp hơn trên log log so với histogram
MAX_POINTS = 300

def sample_ccdf(degrees, max_pts=MAX_POINTS):
    uniq = np.unique(degrees)
    if len(uniq) <= max_pts:
        sampled = uniq
    else:
        idx = np.unique(np.round(np.logspace(0, np.log10(len(uniq) - 1), max_pts)).astype(int))
        sampled = uniq[np.clip(idx, 0, len(uniq) - 1)]
    n = len(degrees)
    ccdf = np.array([(degrees >= k).sum() / n for k in sampled])
    return sampled, ccdf

u_x, u_ccdf = sample_ccdf(user_deg_nonzero)
p_x, p_ccdf = sample_ccdf(prod_deg_nonzero)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.loglog(u_x, u_ccdf, marker='.', markersize=3, linewidth=1.0, color='#2c6fad',
           label='User behaviors')
ax1.set_xlabel('Degree k')
ax1.set_ylabel('P(Degree at least k)')
ax1.set_title('User Degree Distribution CCDF')
ax1.legend(frameon=False)
ax1.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.4)
sns.despine(ax=ax1)
ax1.annotate('Long-tail region', xy=(u_x[-20], u_ccdf[-20]),
             xytext=(u_x[-20] * 0.15, u_ccdf[-20] * 8),
             fontsize=9)

ax2.loglog(p_x, p_ccdf, marker='.', markersize=3, linewidth=1.0, color='#c0392b',
           label='Product behaviors')
ax2.set_xlabel('Degree k')
ax2.set_ylabel('P(Degree at least k)')
ax2.set_title('Product Degree Distribution CCDF')
ax2.legend(frameon=False)
ax2.grid(True, which='both', linestyle='--', linewidth=0.4, alpha=0.4)
sns.despine(ax=ax2)

plt.suptitle('Degree Distribution on REES46 Behavioral Graph Train Split Log Log Scale',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_degree_distribution.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_degree_distribution.png'))
plt.show()

## 4. Temporal Split Composition

Thay cho biểu đồ phân phối theo tháng cần quét raw CSV, phần này dựng trực tiếp từ dữ liệu đã chia
để minh hoạ cách phân bổ tương tác theo protocol chia theo thời gian:

Train: toàn bộ tương tác view, cart, purchase trước mốc chia.
Val và Test: các cặp purchase được giữ làm ground truth full ranking.

Hình này biện minh cho giao thức đánh giá (không leakage) mà không cần chạy lại pipeline raw.

In [ ]:
splits = ['Train', 'Val', 'Test']
view_vals = np.array([E_view_train, 0, 0])
cart_vals = np.array([E_cart_train, 0, 0])
purchase_vals = np.array([E_purchase_train, n_val, n_test])

colors_beh = {'view': '#2c6fad', 'cart': '#5ba4cf', 'purchase': '#f39c12'}

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(splits))
bottom = np.zeros(len(splits), dtype=float)
for name, vals in [('view', view_vals), ('cart', cart_vals), ('purchase', purchase_vals)]:
    ax.bar(x, vals, bottom=bottom, label=name.capitalize(),
           color=colors_beh[name], edgecolor='white', linewidth=0.5, zorder=3)
    bottom += vals

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel('Number of Interactions log scale')
ax.set_title('Temporal Split Composition\nTrain has full behaviors, Val and Test have held-out purchase pairs')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.4, zorder=0)
ax.legend(frameon=False, loc='upper right')

totals = [E_view_train + E_cart_train + E_purchase_train, n_val, n_test]
for xi, tot in zip(x, totals):
    ax.text(xi, tot * 1.5, f'{tot/1e6:.2f}M' if tot >= 1e6 else f'{tot:,}',
            ha='center', va='bottom', fontsize=9)
ax.set_ylim(top=max(totals) * 6)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig_temporal_split.pdf'))
plt.savefig(os.path.join(FIG_DIR, 'fig_temporal_split.png'))
plt.show()
print('Saved fig_temporal_split pdf and png')

## 5. Summary Statistics Table (for Paper)

In [ ]:
avg_all_per_user = E_behavior_total / N_U
avg_purchase_per_user = E_purchase / N_U
r_view = E_view / E_purchase
r_cart = E_cart / E_purchase

def row(metric, value):
    print(f"{metric} {value}")

print('REES46 heterogeneous graph dataset statistics summary')
row('Metric', 'Value')
row('Users', f'{N_U:,}')
row('Products', f'{N_P:,}')
row('Categories', f'{N_C:,}')
row('Brands', f'{N_B:,}')
row('Total Nodes', f'{N_total:,}')
row('View edges', f'{E_view:,}')
row('Cart edges', f'{E_cart:,}')
row('Purchase edges', f'{E_purchase:,}')
if E_belongsTo is not None: row('BelongsTo edges', f'{E_belongsTo:,}')
if E_producedBy is not None: row('ProducedBy edges', f'{E_producedBy:,}')
row('Total Behavioral Edges', f'{E_behavior_total:,}')
row('Sparsity view', f'{sparsity(E_view):.6f}')
row('Sparsity cart', f'{sparsity(E_cart):.6f}')
row('Sparsity purchase', f'{sparsity(E_purchase):.6f}')
row('Avg interactions per user', f'{avg_all_per_user:.2f}')
row('Avg purchases per user', f'{avg_purchase_per_user:.2f}')
row('View purchase ratio', f'{r_view:.2f}')
row('Cart purchase ratio', f'{r_cart:.2f}')
row('Temporal Val pairs', f'{n_val:,}')
row('Temporal Test pairs', f'{n_test:,}')
row('Unique eval users val', f'{eval_users_val:,}')
row('Unique eval users test', f'{eval_users_test:,}')